In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys 
import torch
from pathlib import Path

from satchange import (
    BCEDiceLoss,
    BinaryChangeMetrics,
)

from satchange.models import FCSiamDiff

from torch.utils.data import DataLoader
from satchange.utils import load_config
from satchange.data import (
    RandomFlipRotate,
    prepare_levir_cd, 
    LEVIRCDDataset,
)
from satchange.experiments import (
    load_fc_siam_diff_experiment
)

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Selected GPU:", torch.cuda.get_device_name(0))

Python executable: .venv/bin/python
Python version: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
PyTorch: 2.12.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
Visible GPU count: 1
Selected GPU: NVIDIA A100-SXM4-80GB


In [4]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

full_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_full.yaml"
)

resized_config_path = (
    project_root
    / "configs"
    / "fc_siam_diff_resized.yaml"
)


print("The project root: ", project_root)

print("Full config:", full_config_path)
print("Full config exists:", full_config_path.exists())

print("Resized config:", resized_config_path)
print("Resized config exists:", resized_config_path.exists())

The project root:  .
Full config: configs/fc_siam_diff_full.yaml
Full config exists: True
Resized config: configs/fc_siam_diff_resized.yaml
Resized config exists: True


### Resized Dataset

In [5]:
EXPERIMENT = "resized"
# Available "full" and "resized"


config, config_path = load_fc_siam_diff_experiment(
    experiment=EXPERIMENT,
    project_root=project_root,
)

print("Experiment:", EXPERIMENT)
print("Configuration:", config_path)
print("Image size:", config["data"]["image_size"])
print("Batch size:", config["data"]["batch_size"])
print("Output directory:", config["output"]["directory"])

Experiment: resized
Configuration: configs/fc_siam_diff_resized.yaml
Image size: 256
Batch size: 16
Output directory: artifacts/fc_siam_diff_resized


In [6]:
# path to the dataset
dataset_root = prepare_levir_cd(
    config=config,
    project_root=project_root,
)

print("Ready dataset root:", dataset_root)

Download directory is not empty: data/downloads/levir_cd
Skipping download.
Split is already extracted: data/LEVIR-CD/train
Skipping extraction.
Split is already extracted: data/LEVIR-CD/val
Skipping extraction.
Split is already extracted: data/LEVIR-CD/test
Skipping extraction.

Split:         train
Before images: 445
After images:  445
Labels:        445

Split:         val
Before images: 64
After images:  64
Labels:        64

Split:         test
Before images: 128
After images:  128
Labels:        128

Total samples: 637
LEVIR-CD structure is valid.
Ready dataset root: data/LEVIR-CD


In [7]:
train_transform = RandomFlipRotate(
    horizontal_flip_probability = 0.5,
    vertical_flip_probability = 0.5,
)

train_dataset = LEVIRCDDataset(
    dataset_root=dataset_root, 
    split="train",
    image_size=config["data"]["image_size"],
    transform=train_transform
)

In [8]:
validation_dataset = LEVIRCDDataset(
    dataset_root=dataset_root,
    split="val",
    image_size=config["data"]["image_size"],
    transform=None,
)

test_dataset = LEVIRCDDataset(
    dataset_root=dataset_root,
    split="test",
    image_size=config["data"]["image_size"],
    transform=None,
)

In [9]:
num_workers = config["data"]["num_workers"]
batch_size = config["data"]["batch_size"]

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

In [10]:
def calculate_positive_weight(
    data_loader: DataLoader,
) -> tuple[float, int, int]:
    """Calculate the negative-to-positive pixel ratio."""

    positive_pixels = 0
    total_pixels = 0

    for batch in data_loader:
        masks = batch["mask"]

        if masks.ndim != 4:
            raise ValueError(
                "Masks must have shape [B, C, H, W]. "
                f"Received {masks.shape}."
            )

        target_mask = masks >= 0.5

        positive_pixels += int(
            target_mask.sum().item()
        )

        total_pixels += target_mask.numel()

    if total_pixels == 0:
        raise RuntimeError(
            "The training data loader contains no pixels."
        )

    negative_pixels = (
        total_pixels - positive_pixels
    )

    if positive_pixels == 0:
        raise RuntimeError(
            "The training split contains no changed pixels."
        )

    positive_weight = (
        negative_pixels / positive_pixels
    )

    return (
        positive_weight,
        positive_pixels,
        negative_pixels,
    )


(
    positive_weight,
    positive_pixels,
    negative_pixels,
) = calculate_positive_weight(
    train_loader
)

print("Changed training pixels:", positive_pixels)
print("Unchanged training pixels:", negative_pixels)

print(
    "Changed fraction:",
    positive_pixels
    / (positive_pixels + negative_pixels),
)

print(
    "Calculated positive weight:",
    positive_weight,
)

Changed training pixels: 1338323
Unchanged training pixels: 27825197
Changed fraction: 0.04589031090897121
Calculated positive weight: 20.791092284896845


In [11]:
train_batch = next(iter(train_loader))

print("Before:", train_batch["before"].shape)
print("After:", train_batch["after"].shape)
print("Mask:", train_batch["mask"].shape)
print("Sample IDs:", train_batch["sample_id"])

Before: torch.Size([16, 3, 256, 256])
After: torch.Size([16, 3, 256, 256])
Mask: torch.Size([16, 1, 256, 256])
Sample IDs: ['train_89', 'train_19', 'train_319', 'train_150', 'train_69', 'train_186', 'train_376', 'train_82', 'train_361', 'train_140', 'train_396', 'train_435', 'train_164', 'train_395', 'train_381', 'train_365']


In [12]:
before = train_batch["before"].to(
    device,
    non_blocking=True,
)

after = train_batch["after"].to(
    device,
    non_blocking=True,
)

mask = train_batch["mask"].to(
    device,
    dtype=torch.float32,
    non_blocking=True,
)

In [13]:
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


model = FCSiamDiff(
    in_channels=(
        config["model"]["parameters"]["in_channels"]
    ),
    out_channels=(
        config["model"]["parameters"]["out_channels"]
    ),
    base_channels=(
        config["model"]["parameters"]["base_channels"]
    ),
).to(device)


loss_function = BCEDiceLoss(
    bce_weight=(
        config["training"]["loss"]["bce_weight"]
    ),
    dice_weight=(
        config["training"]["loss"]["dice_weight"]
    ),
    pos_weight=5,
).to(device)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["training"]["learning_rate"],
    weight_decay=(
        config["training"]["weight_decay"]
    ),
)


print(
    "BCE positive weight:",
    loss_function.bce_loss.pos_weight,
)

BCE positive weight: tensor([[[20.7911]]], device='cuda:0')


In [14]:
def train_one_epoch(
    model: torch.nn.Module,
    data_loader: DataLoader,
    loss_function: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    """Train the model for one complete epoch."""
    
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in data_loader:
        before = batch["before"].to(
            device,
            non_blocking=True,
        )
        after = batch["after"].to(
            device,
            non_blocking=True,
        )
        mask = batch["mask"].to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True,
        )

        logits = model(
            before,
            after,
        )

        loss = loss_function(
            logits,
            mask,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Training loss is not finite: {loss.item()}"
            )

        loss.backward()
        optimizer.step()

        current_batch_size = before.shape[0]

        total_loss += (
            loss.item() * current_batch_size
        )
        total_samples += current_batch_size

    if total_samples == 0:
        raise RuntimeError("Training data loader is empty.")    

    return total_loss / total_samples

In [15]:
def validate_one_epoch(
    model: torch.nn.Module,
    data_loader: DataLoader,
    loss_function: torch.nn.Module,
    device: torch.device,
    threshold: float,
) -> float:
    """Evaluate the model for one complete validation epoch."""

    model.eval()

    total_loss = 0.0
    total_samples = 0

    metric_tracker = BinaryChangeMetrics(
        threshold=threshold,
    )

    with torch.no_grad():
        for batch in data_loader:
            before = batch["before"].to(
            device,
            non_blocking=True,
            )

            after = batch["after"].to(
                device,
                non_blocking=True,
            )

            mask = batch["mask"].to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            logits = model(
                before,
                after,
            )

            loss = loss_function(
                logits, 
                mask,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Validation loss is not finite: {loss.item()}"
                )

            current_batch_size = before.shape[0]

            total_loss += (
                loss.item() * current_batch_size
            )
            total_samples += current_batch_size

            metric_tracker.update(
                logits,
                mask,
            )
        
        if total_samples == 0:
            raise RuntimeError(
                "Validation data loader is empty."
            )

        results = metric_tracker.compute()

        results["loss"] = (
            total_loss / total_samples
        )

        return {
            "results": results,
            "logits": logits,
        }
        

In [16]:
# train_loss = train_one_epoch(
#     model=model,
#     data_loader=train_loader,
#     loss_function=loss_function,
#     optimizer=optimizer,
#     device=device,
# )

# validation_results = validate_one_epoch(
#     model=model,
#     data_loader=validation_loader,
#     loss_function=loss_function,
#     device=device,
#     threshold=config["training"]["threshold"],
# )

In [17]:
def train_one_epoch_checked(
    model: torch.nn.Module,
    data_loader,
    loss_function: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    """Train the model for one complete epoch."""

    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in data_loader:
        before = batch["before"].to(
            device,
            non_blocking=True,
        )

        after = batch["after"].to(
            device,
            non_blocking=True,
        )

        mask = batch["mask"].to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True,
        )

        logits = model(
            before,
            after,
        )

        loss = loss_function(
            logits,
            mask,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Training loss is not finite: {loss.item()}"
            )

        loss.backward()
        optimizer.step()

        current_batch_size = before.shape[0]

        total_loss += (
            loss.item() * current_batch_size
        )

        total_samples += current_batch_size

    if total_samples == 0:
        raise RuntimeError(
            "Training data loader is empty."
        )

    return total_loss / total_samples


@torch.no_grad()
def evaluate_thresholds(
    model: torch.nn.Module,
    data_loader,
    loss_function: torch.nn.Module,
    device: torch.device,
    thresholds: list[float],
) -> dict:
    """Evaluate one fixed model at multiple probability thresholds."""

    if not thresholds:
        raise ValueError(
            "At least one threshold is required."
        )

    for threshold in thresholds:
        if not 0.0 < threshold < 1.0:
            raise ValueError(
                "Every threshold must be between 0 and 1. "
                f"Received {threshold}."
            )

    model.eval()

    total_loss = 0.0
    total_samples = 0

    confusion_counts = {
        threshold: {
            "tp": 0,
            "fp": 0,
            "fn": 0,
            "tn": 0,
        }
        for threshold in thresholds
    }

    changed_probability_sum = 0.0
    unchanged_probability_sum = 0.0

    changed_pixel_count = 0
    unchanged_pixel_count = 0

    probability_minimum = float("inf")
    probability_maximum = float("-inf")

    changed_probability_maximum = float("-inf")
    unchanged_probability_maximum = float("-inf")

    for batch in data_loader:
        before = batch["before"].to(
            device,
            non_blocking=True,
        )

        after = batch["after"].to(
            device,
            non_blocking=True,
        )

        mask = batch["mask"].to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        logits = model(
            before,
            after,
        )

        loss = loss_function(
            logits,
            mask,
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Validation loss is not finite: {loss.item()}"
            )

        current_batch_size = before.shape[0]

        total_loss += (
            loss.item() * current_batch_size
        )

        total_samples += current_batch_size

        probabilities = torch.sigmoid(
            logits
        )

        target_mask = mask >= 0.5

        changed_probabilities = probabilities[
            target_mask
        ]

        unchanged_probabilities = probabilities[
            ~target_mask
        ]

        changed_probability_sum += (
            changed_probabilities.sum().item()
        )

        unchanged_probability_sum += (
            unchanged_probabilities.sum().item()
        )

        changed_pixel_count += (
            changed_probabilities.numel()
        )

        unchanged_pixel_count += (
            unchanged_probabilities.numel()
        )

        probability_minimum = min(
            probability_minimum,
            probabilities.min().item(),
        )

        probability_maximum = max(
            probability_maximum,
            probabilities.max().item(),
        )

        if changed_probabilities.numel() > 0:
            changed_probability_maximum = max(
                changed_probability_maximum,
                changed_probabilities.max().item(),
            )

        if unchanged_probabilities.numel() > 0:
            unchanged_probability_maximum = max(
                unchanged_probability_maximum,
                unchanged_probabilities.max().item(),
            )

        for threshold in thresholds:
            predictions = (
                probabilities >= threshold
            )

            tp = torch.logical_and(
                predictions,
                target_mask,
            ).sum()

            fp = torch.logical_and(
                predictions,
                ~target_mask,
            ).sum()

            fn = torch.logical_and(
                ~predictions,
                target_mask,
            ).sum()

            tn = torch.logical_and(
                ~predictions,
                ~target_mask,
            ).sum()

            confusion_counts[threshold]["tp"] += int(
                tp.item()
            )

            confusion_counts[threshold]["fp"] += int(
                fp.item()
            )

            confusion_counts[threshold]["fn"] += int(
                fn.item()
            )

            confusion_counts[threshold]["tn"] += int(
                tn.item()
            )

    if total_samples == 0:
        raise RuntimeError(
            "Validation data loader is empty."
        )

    threshold_results = []

    for threshold in thresholds:
        counts = confusion_counts[threshold]

        tp = counts["tp"]
        fp = counts["fp"]
        fn = counts["fn"]
        tn = counts["tn"]

        precision_denominator = tp + fp
        recall_denominator = tp + fn
        f1_denominator = 2 * tp + fp + fn
        iou_denominator = tp + fp + fn
        total_pixels = tp + fp + fn + tn

        precision = (
            tp / precision_denominator
            if precision_denominator > 0
            else 0.0
        )

        recall = (
            tp / recall_denominator
            if recall_denominator > 0
            else 0.0
        )

        f1 = (
            2 * tp / f1_denominator
            if f1_denominator > 0
            else 0.0
        )

        iou = (
            tp / iou_denominator
            if iou_denominator > 0
            else 0.0
        )

        accuracy = (
            (tp + tn) / total_pixels
            if total_pixels > 0
            else 0.0
        )

        threshold_results.append(
            {
                "threshold": threshold,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "iou": iou,
                "accuracy": accuracy,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                "predicted_changed_pixels": tp + fp,
            }
        )

    mean_changed_probability = (
        changed_probability_sum
        / changed_pixel_count
        if changed_pixel_count > 0
        else 0.0
    )

    mean_unchanged_probability = (
        unchanged_probability_sum
        / unchanged_pixel_count
        if unchanged_pixel_count > 0
        else 0.0
    )

    return {
        "loss": total_loss / total_samples,
        "probability_minimum": probability_minimum,
        "probability_maximum": probability_maximum,
        "mean_changed_probability": mean_changed_probability,
        "mean_unchanged_probability": mean_unchanged_probability,
        "maximum_changed_probability": changed_probability_maximum,
        "maximum_unchanged_probability": unchanged_probability_maximum,
        "changed_pixel_count": changed_pixel_count,
        "unchanged_pixel_count": unchanged_pixel_count,
        "threshold_results": threshold_results,
    }

In [30]:
import copy


EXPERIMENT_SEED = 42
PILOT_EPOCHS = 60

POSITIVE_WEIGHTS = [10]

THRESHOLDS = [
    round(value / 100, 2)
    for value in range(5, 96, 5)
]
# 0.05, 0.10, 0.15, ..., 0.90, 0.95


# Create one initial model state.
# Every experiment will begin from exactly these parameters.
torch.manual_seed(EXPERIMENT_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        EXPERIMENT_SEED
    )


initial_model = FCSiamDiff(
    in_channels=(
        config["model"]["parameters"]["in_channels"]
    ),
    out_channels=(
        config["model"]["parameters"]["out_channels"]
    ),
    base_channels=(
        config["model"]["parameters"]["base_channels"]
    ),
)

initial_model_state = copy.deepcopy(
    initial_model.state_dict()
)

del initial_model


output_directory = (
    Path(project_root)
    / config["output"]["directory"]
    / "pos_weight_search"
)

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)


all_experiment_results = []


for current_positive_weight in POSITIVE_WEIGHTS:
    if current_positive_weight is None:
        experiment_name = "unweighted"
    else:
        experiment_name = (
            f"pos_weight_{current_positive_weight:g}"
        )

    print()
    print("#" * 100)
    print(f"Experiment: {experiment_name}")
    print("#" * 100)

    # Create a new DataLoader with the same shuffle seed.
    loader_generator = torch.Generator()
    loader_generator.manual_seed(
        EXPERIMENT_SEED
    )

    current_train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        generator=loader_generator,
    )

    # Create a completely new model.
    model = FCSiamDiff(
        in_channels=(
            config["model"]["parameters"]["in_channels"]
        ),
        out_channels=(
            config["model"]["parameters"]["out_channels"]
        ),
        base_channels=(
            config["model"]["parameters"]["base_channels"]
        ),
    )

    # Give every experiment exactly the same
    # initial model parameters.
    model.load_state_dict(
        initial_model_state
    )

    model = model.to(device)

    # Create a completely new loss function.
    loss_function = BCEDiceLoss(
        bce_weight=(
            config["training"]["loss"]["bce_weight"]
        ),
        dice_weight=(
            config["training"]["loss"]["dice_weight"]
        ),
        pos_weight=current_positive_weight,
    ).to(device)

    # Create a completely new optimizer.
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["training"]["learning_rate"],
        weight_decay=(
            config["training"]["weight_decay"]
        ),
    )

    # Reset random augmentation state before training.
    torch.manual_seed(EXPERIMENT_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            EXPERIMENT_SEED
        )

    experiment_history = []

    best_epoch = None
    best_threshold = None
    best_precision = None
    best_recall = None
    best_f1 = -1.0
    best_iou = -1.0
    best_validation_loss = None
    best_model_state = None

    for epoch in range(
        1,
        PILOT_EPOCHS + 1,
    ):
        train_loss = train_one_epoch_checked(
            model=model,
            data_loader=current_train_loader,
            loss_function=loss_function,
            optimizer=optimizer,
            device=device,
        )

        validation_results = evaluate_thresholds(
            model=model,
            data_loader=validation_loader,
            loss_function=loss_function,
            device=device,
            thresholds=THRESHOLDS,
        )

        best_threshold_result = max(
            validation_results["threshold_results"],
            key=lambda result: result["f1"],
        )

        threshold_05_result = next(
            result
            for result
            in validation_results["threshold_results"]
            if result["threshold"] == 0.50
        )

        mean_changed_probability = (
            validation_results[
                "mean_changed_probability"
            ]
        )

        mean_unchanged_probability = (
            validation_results[
                "mean_unchanged_probability"
            ]
        )

        probability_gap = (
            mean_changed_probability
            - mean_unchanged_probability
        )

        epoch_result = {
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": (
                validation_results["loss"]
            ),
            "best_threshold": (
                best_threshold_result["threshold"]
            ),
            "precision": (
                best_threshold_result["precision"]
            ),
            "recall": (
                best_threshold_result["recall"]
            ),
            "f1": best_threshold_result["f1"],
            "iou": best_threshold_result["iou"],
            "f1_at_05": threshold_05_result["f1"],
            "mean_changed_probability": (
                mean_changed_probability
            ),
            "mean_unchanged_probability": (
                mean_unchanged_probability
            ),
            "probability_gap": probability_gap,
        }

        experiment_history.append(
            epoch_result
        )

        print()
        print("=" * 100)
        print(
            f"Epoch {epoch}/{PILOT_EPOCHS}"
        )
        print("=" * 100)

        print(
            f"Train loss:       {train_loss:.6f}"
        )

        print(
            "Validation loss:  "
            f"{validation_results['loss']:.6f}"
        )

        print(
            "Best threshold:   "
            f"{best_threshold_result['threshold']:.2f}"
        )

        print(
            "Precision:        "
            f"{best_threshold_result['precision']:.6f}"
        )

        print(
            "Recall:           "
            f"{best_threshold_result['recall']:.6f}"
        )

        print(
            "F1:               "
            f"{best_threshold_result['f1']:.6f}"
        )

        print(
            "IoU:              "
            f"{best_threshold_result['iou']:.6f}"
        )

        print(
            "F1 at threshold 0.50: "
            f"{threshold_05_result['f1']:.6f}"
        )

        print(
            "Mean P(change):    "
            f"{mean_changed_probability:.6f}"
        )

        print(
            "Mean P(no change): "
            f"{mean_unchanged_probability:.6f}"
        )

        print(
            "Probability gap:   "
            f"{probability_gap:.6f}"
        )

        # Save the best epoch according to validation F1.
        if best_threshold_result["f1"] > best_f1:
            best_epoch = epoch

            best_threshold = (
                best_threshold_result["threshold"]
            )

            best_precision = (
                best_threshold_result["precision"]
            )

            best_recall = (
                best_threshold_result["recall"]
            )

            best_f1 = (
                best_threshold_result["f1"]
            )

            best_iou = (
                best_threshold_result["iou"]
            )

            best_validation_loss = (
                validation_results["loss"]
            )

            # Copy checkpoint tensors to CPU.
            best_model_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter
                in model.state_dict().items()
            }

    checkpoint_path = (
        output_directory
        / f"{experiment_name}_best.pt"
    )

    torch.save(
        {
            "experiment_name": experiment_name,
            "positive_weight": current_positive_weight,
            "best_epoch": best_epoch,
            "best_threshold": best_threshold,
            "best_precision": best_precision,
            "best_recall": best_recall,
            "best_f1": best_f1,
            "best_iou": best_iou,
            "best_validation_loss": best_validation_loss,
            "model_state_dict": best_model_state,
            "config": config,
            "history": experiment_history,
        },
        checkpoint_path,
    )

    all_experiment_results.append(
        {
            "experiment_name": experiment_name,
            "positive_weight": current_positive_weight,
            "best_epoch": best_epoch,
            "best_threshold": best_threshold,
            "precision": best_precision,
            "recall": best_recall,
            "f1": best_f1,
            "iou": best_iou,
            "checkpoint_path": checkpoint_path,
            "history": experiment_history,
        }
    )

    print()
    print("-" * 100)

    print(
        f"Best result for {experiment_name}:"
    )

    print(
        f"Epoch:     {best_epoch}"
    )

    print(
        f"Threshold: {best_threshold:.2f}"
    )

    print(
        f"Precision: {best_precision:.6f}"
    )

    print(
        f"Recall:    {best_recall:.6f}"
    )

    print(
        f"F1:        {best_f1:.6f}"
    )

    print(
        f"IoU:       {best_iou:.6f}"
    )

    print(
        f"Checkpoint: {checkpoint_path}"
    )

    del model
    del loss_function
    del optimizer
    del current_train_loader

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


####################################################################################################
Experiment: pos_weight_10
####################################################################################################

Epoch 1/60
Train loss:       0.917633
Validation loss:  0.891457
Best threshold:   0.35
Precision:        0.045725
Recall:           0.982839
F1:               0.087384
IoU:              0.045688
F1 at threshold 0.50: 0.000000
Mean P(change):    0.361054
Mean P(no change): 0.357121
Probability gap:   0.003934

Epoch 2/60
Train loss:       0.903707
Validation loss:  0.860251
Best threshold:   0.40
Precision:        0.157221
Recall:           0.284716
F1:               0.202578
IoU:              0.112705
F1 at threshold 0.50: 0.034546
Mean P(change):    0.372008
Mean P(no change): 0.322354
Probability gap:   0.049654

Epoch 3/60
Train loss:       0.897704
Validation loss:  0.865830
Best threshold:   0.35
Precision:        0.143615
Recall:           0.321146
F1: 

In [29]:
sorted_results = sorted(
    all_experiment_results,
    key=lambda result: result["f1"],
    reverse=True,
)


print()
print("=" * 100)

print(
    "Experiment      | Epoch | Threshold | "
    "Precision | Recall | F1     | IoU"
)

print("=" * 100)

for result in sorted_results:
    print(
        f"{result['experiment_name']:15s} | "
        f"{result['best_epoch']:5d} | "
        f"{result['best_threshold']:9.2f} | "
        f"{result['precision']:9.4f} | "
        f"{result['recall']:6.4f} | "
        f"{result['f1']:6.4f} | "
        f"{result['iou']:6.4f}"
    )


best_experiment = sorted_results[0]

print()
print("Best experiment by validation F1")

print(
    "Experiment:",
    best_experiment["experiment_name"],
)

print(
    "Positive weight:",
    best_experiment["positive_weight"],
)

print(
    "Best epoch:",
    best_experiment["best_epoch"],
)

print(
    "Best threshold:",
    best_experiment["best_threshold"],
)

print(
    "Precision:",
    f"{best_experiment['precision']:.6f}",
)

print(
    "Recall:",
    f"{best_experiment['recall']:.6f}",
)

print(
    "F1:",
    f"{best_experiment['f1']:.6f}",
)

print(
    "IoU:",
    f"{best_experiment['iou']:.6f}",
)

print(
    "Checkpoint:",
    best_experiment["checkpoint_path"],
)


Experiment      | Epoch | Threshold | Precision | Recall | F1     | IoU
pos_weight_10   |    20 |      0.70 |    0.3522 | 0.5059 | 0.4153 | 0.2620

Best experiment by validation F1
Experiment: pos_weight_10
Positive weight: 10
Best epoch: 20
Best threshold: 0.7
Precision: 0.352164
Recall: 0.505899
F1: 0.415260
IoU: 0.262037
Checkpoint: artifacts/fc_siam_diff_resized/pos_weight_search/pos_weight_10_best.pt


In [ ]:
# before_features = model._encode(before)
# after_features = model._encode(after)

# feature_differences = model._compute_feature_differences(
#     before_features,
#     after_features,
# )

# decoded_features = model._decode(
#     feature_differences
# )

# print(decoded_features.shape)

In [ ]:
# # switches the model into training mode
# model.train()
# # and
# model.eval()

# for batch in train_loader:
#     # cleares gradients left from previous batches 'cause PyTorch accumulates gradients
#     optimizer.zero_grad()

#     # the prediction and computation graph
#     prediction = model(
#         batch["before"],
#         batch["after"],
#     )

#     # computing the loss (how different the prediction from the real mask)
#     loss = loss_function(
#         prediction,
#         batch["mask"],
#     )

#     # computing the gradient (what change to reduce the loss)
#     loss.backward()

#     # update the weights using gradients
#     optimizer.step()


# # PyTorch cycle that does not effect the model. It's used to evaluate but not to train.
# with torch.no_grad():
#     for batch in validation_loader:
#         prediction = model(
#             batch["before"],
#             batch["after"],
#         )

#         # computing the loss
#         loss = loss_function(
#             prediction,
#             batch["mask"]
#         )

### Full sized Dataset

In [ ]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [ ]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")

In [ ]:
# before_features = model._encode(before)
# after_features = model._encode(after)

# feature_differences = model._compute_feature_differences(
#     before_features,
#     after_features,
# )

# decoded_features = model._decode(
#     feature_differences
# )

# print(decoded_features.shape)

In [ ]:
# # switches the model into training mode
# model.train()
# # and
# model.eval()

# for batch in train_loader:
#     # cleares gradients left from previous batches 'cause PyTorch accumulates gradients
#     optimizer.zero_grad()

#     # the prediction and computation graph
#     prediction = model(
#         batch["before"],
#         batch["after"],
#     )

#     # computing the loss (how different the prediction from the real mask)
#     loss = loss_function(
#         prediction,
#         batch["mask"],
#     )

#     # computing the gradient (what change to reduce the loss)
#     loss.backward()

#     # update the weights using gradients
#     optimizer.step()


# # PyTorch cycle that does not effect the model. It's used to evaluate but not to train.
# with torch.no_grad():
#     for batch in validation_loader:
#         prediction = model(
#             batch["before"],
#             batch["after"],
#         )

#         # computing the loss
#         loss = loss_function(
#             prediction,
#             batch["mask"]
#         )

### Full sized Dataset

In [ ]:
# EXPERIMENT = "full"

# config, config_path = load_fc_siam_diff_experiment(
#     experiment=EXPERIMENT,
#     project_root=project_root,
# )

# print(f"Configuration {config_path}")
# print(f"Data settings: {config['data']}")

In [ ]:
# dataset_root = prepare_levir_cd(
#     config=config,
#     project_root=project_root,
# )


# dataset = LEVIRCDDataset(
#     dataset_root=dataset_root,
#     split="train",
#     image_size=config["data"]["image_size"],
# )

# sample=dataset[0]


# print(f"Number of samples: {len(dataset)}")
# print(f"Sample ID: {sample['sample_id']}")
# print(f"Before shape: {sample['before'].shape}")
# print(f"After shape: {sample['after'].shape}")
# print(f"Mask shape: {sample['mask'].shape}")